## 1. Cargar los datos y mostrar las primeras filas

Importamos Pandas con el alias `pd` y utilizamos `pd.read_csv("results.csv")`
para leer los resultados del simulador.

Guardamos los datos en `results`, un **DataFrame**: una tabla en memoria
organizada en filas y columnas.

- Cada fila representa una tirada.
- Cada columna contiene una característica, como la apuesta o el saldo.
- El índice identifica las filas y, en esta carga, comienza en cero.

`results.head()` muestra las primeras cinco filas sin eliminar ni modificar
el resto de los datos.

In [34]:
import pandas as pd

results = pd.read_csv("results.csv")

results.head()

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after
0,martingala,1,par,30,10,100,1,10,110
1,martingala,2,par,0,10,110,0,-10,100
2,martingala,3,par,18,20,100,1,20,120
3,martingala,4,par,0,10,120,0,-10,110
4,martingala,5,par,10,20,110,1,20,130


## 2. Consultar el tamaño de los datos

`results.shape` devuelve una tupla con dos valores: `(filas, columnas)`.

En este archivo, el resultado es `(35, 9)`: tenemos 35 tiradas y 9 columnas.
El índice no cuenta como una columna.

`shape` es un atributo, por lo que se consulta sin paréntesis.

In [35]:
results.shape

(35, 9)

## 3. Consultar los nombres de las columnas

`results.columns` devuelve las etiquetas de las columnas.

Nos permite conocer sus nombres exactos para seleccionarlas y utilizarlas
en operaciones posteriores. Por ejemplo, `results["profit"]` selecciona
la columna del beneficio.

In [36]:
results.columns

Index(['strategy', 'spin', 'side', 'number', 'bet', 'balance_before', 'won',
       'profit', 'balance_after'],
      dtype='str')

## 4. Revisar los tipos de datos

`results.dtypes` muestra el tipo de datos de cada columna.

En este archivo:

- `strategy` y `side` contienen texto (`str`).
- Las otras siete columnas contienen números enteros (`int64`).

Los tipos influyen en las operaciones que podemos realizar. Necesitamos
que las apuestas, los saldos y los beneficios se interpreten como números
para calcular con ellos.

La columna `won` contiene enteros que representan el resultado:
`1` significa ganar y `0` significa perder.

El `dtype: object` que aparece al final describe la Serie devuelta por
`dtypes`, no el tipo de todas las columnas del DataFrame.

In [37]:
results.dtypes

strategy            str
spin              int64
side                str
number            int64
bet               int64
balance_before    int64
won               int64
profit            int64
balance_after     int64
dtype: object

## 5. Obtener un resumen del DataFrame

`results.info()` muestra:

- El número de filas y su índice.
- Los nombres de las columnas.
- La cantidad de valores no ausentes de cada columna.
- Los tipos de datos.
- Una estimación de la memoria utilizada.

En nuestro archivo, todas las columnas muestran `35 non-null`.
Esto significa que Pandas no ha identificado valores ausentes en ellas.

Que un dato esté presente no garantiza que sea correcto:
también debemos comprobar sus valores y su coherencia.

In [38]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   strategy        35 non-null     str  
 1   spin            35 non-null     int64
 2   side            35 non-null     str  
 3   number          35 non-null     int64
 4   bet             35 non-null     int64
 5   balance_before  35 non-null     int64
 6   won             35 non-null     int64
 7   profit          35 non-null     int64
 8   balance_after   35 non-null     int64
dtypes: int64(7), str(2)
memory usage: 2.6 KB


## 6. Contar los valores ausentes

`results.isna()` comprueba cada celda y devuelve:

- `True` si el valor está ausente.
- `False` si el valor está presente.

Al aplicar `.sum()`, contamos los valores ausentes de cada columna:
`True` cuenta como 1 y `False` como 0.

Todas las columnas han dado cero, por lo que no hay valores ausentes
reconocidos por Pandas.

Un valor cero no es un dato ausente. Por ejemplo, `won = 0` representa
una pérdida y es un dato válido.

In [39]:
results.isna().sum()

strategy          0
spin              0
side              0
number            0
bet               0
balance_before    0
won               0
profit            0
balance_after     0
dtype: int64

## 7. Seleccionar una columna

`results["number"]` selecciona la columna que contiene el número obtenido
en cada tirada.

El resultado es una **Serie** de Pandas: una estructura de una dimensión
que conserva los valores de la columna y sus índices.

Seleccionar una columna nos permite aplicar comprobaciones u operaciones
sobre sus valores.

In [40]:
results["number"]

0     30
1      0
2     18
3      0
4     10
5     22
6     10
7      6
8     36
9      6
10    18
11    22
12    12
13     8
14     4
15     0
16    21
17    28
18    35
19    27
20    15
21    34
22    17
23    35
24    20
25    18
26    18
27    23
28    22
29    29
30    14
31    17
32    15
33    23
34    19
Name: number, dtype: int64

## 8. Detectar números fuera del rango permitido

En una ruleta europea, los números posibles van del 0 al 36, ambos incluidos.

Buscamos filas donde `number` sea menor que 0 o mayor que 36.

- `<` y `>` comparan cada valor con los límites.
- `|` combina las condiciones con un «o».
- Los paréntesis agrupan cada comparación.
- `results[...]` selecciona las filas donde la condición es verdadera.

Guardamos las filas detectadas en `invalid_numbers`.

El resultado está vacío: no hay números fuera del rango permitido.
Este filtro no modifica los datos originales.

In [41]:
invalid_numbers = results[
    (results["number"] < 0) | (results["number"] > 36)
]

invalid_numbers

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after


## 9. Comprobar el saldo de cada tirada

El saldo debe cumplir esta igualdad:

**Saldo posterior = saldo anterior + beneficio de la tirada**

Si el beneficio es negativo, sumarlo reduce el saldo.
Por ejemplo: `110 + (-10) = 100`.

Calculamos `expected_balance` sumando las columnas `balance_before`
y `profit`. Pandas realiza la operación entre los valores correspondientes
de cada fila, sin necesidad de escribir un bucle.

Después seleccionamos las filas cuyo `balance_after` sea distinto
del saldo calculado. El operador `!=` significa «distinto de».

El resultado de `inconsistent_balances` está vacío:
la igualdad se cumple en las 35 tiradas.

In [42]:
expected_balance = results["balance_before"] + results["profit"]

inconsistent_balances = results[
    results["balance_after"] != expected_balance
]

inconsistent_balances

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after


## 10. Validar los valores del resultado

La columna `won` solo debe contener dos valores:

- `0`: la apuesta se pierde.
- `1`: la apuesta se gana.

`.isin([0, 1])` comprueba si cada valor pertenece a la lista permitida.
El operador `~` invierte el resultado, de forma que seleccionamos
los valores que no pertenecen a esa lista.

El resultado de `invalid_outcomes` está vacío:
todos los valores son 0 o 1.

Esta comprobación valida los valores permitidos, pero no comprueba
si el resultado corresponde al número que salió y al lado apostado.

In [43]:
invalid_outcomes = results[
    ~results["won"].isin([0, 1])
]
invalid_outcomes

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after


## 11. Comprobar el beneficio de cada tirada

Para las apuestas a par o impar del simulador:

- Si ganamos, el beneficio neto debe ser igual a la apuesta.
- Si perdemos, el beneficio debe ser igual a la apuesta en negativo.

Buscamos dos posibles errores:

1. `won` vale 1, pero `profit` es distinto de `bet`.
2. `won` vale 0, pero `profit` es distinto de `-bet`.

El operador `&` significa «y»: exige que se cumplan ambas condiciones.
El operador `|` significa «o»: basta con que se detecte uno de los errores.

El resultado de `inconsistent_profits` está vacío:
el beneficio coincide con la apuesta y con el resultado registrado
en todas las tiradas.

In [44]:
inconssistent_profits = results[
    ((results["won"] == 1) & (results["profit"] != results["bet"]))
    | ((results["won"] == 0) & (results["profit"] != -results["bet"]))
]

inconssistent_profits

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after


## 12. Validar las cantidades apostadas

Una apuesta válida en este simulador debe:

- Ser mayor que cero.
- No superar el saldo disponible antes de apostar.

Seleccionamos las filas donde `bet <= 0` o donde
`bet > balance_before`.

El resultado de `invalid_bets` está vacío:
todas las apuestas son positivas y ninguna supera el saldo disponible.

In [45]:
invalid_bets = results[
    (results["bet"] <= 0)
    | (results["bet"] > results["balance_before"])
]

invalid_bets

,strategy,spin,side,number,bet,balance_before,won,profit,balance_after


## 13. Seleccionar las tiradas con ganancias

Filtramos las filas donde `profit > 0` y las guardamos en `winning_spins`.

Después mostramos únicamente estas columnas:

- `spin`: número de tirada.
- `bet`: cantidad apostada.
- `profit`: beneficio de la tirada.
- `balance_after`: saldo posterior.

Para seleccionar varias columnas usamos una lista de nombres dentro
de los corchetes de selección: `df[["column_a", "column_b"]]`.

El resultado contiene 20 tiradas ganadoras y conserva los índices originales.
Ganar una tirada no significa necesariamente que la sesión completa
termine con beneficios.

In [46]:
winning_spins = results[results["profit"] > 0]
winning_spins[["spin", "bet", "profit", "balance_after"]]

,spin,bet,profit,balance_after
0,1,10,10,110
2,3,20,20,120
4,5,20,20,130
5,6,10,10,140
6,7,10,10,150
7,8,10,10,160
8,9,10,10,170
9,10,10,10,180
10,11,10,10,190
11,12,10,10,200


## 14. Seleccionar las tiradas con pérdidas

Filtramos las filas donde `profit < 0` y las guardamos en `losing_spins`.

Mostramos las mismas columnas que en las tiradas ganadoras para facilitar
su comparación.

El resultado contiene 15 tiradas perdedoras.

Las cuatro últimas tiradas tienen pérdidas de 10, 20, 40 y 80:
una pérdida conjunta de 150. El saldo pasa de 300 a 150.

In [47]:
losing_spins = results[results["profit"] < 0]

losing_spins[["spin", "bet", "profit", "balance_after"]]

,spin,bet,profit,balance_after
1,2,10,-10,100
3,4,10,-10,110
15,16,10,-10,220
16,17,20,-20,200
18,19,10,-10,230
19,20,20,-20,210
20,21,40,-40,170
22,23,10,-10,240
23,24,20,-20,220
27,28,10,-10,270


## 15. Comprobar el beneficio total de la sesión

Para este archivo, que estamos analizando como una única sesión
ordenada por tiradas, calculamos el beneficio por dos caminos:

1. Sumamos los valores de `profit`, incluidas las pérdidas.
2. Restamos el saldo inicial al saldo final.

Utilizamos:

- `.iloc[0]` para obtener el primer valor por posición.
- `.iloc[-1]` para obtener el último valor por posición.
- `.sum()` para sumar los beneficios de todas las tiradas.
- `==` para comprobar si los dos cálculos coinciden.

Los resultados obtenidos son:

| Concepto | Resultado |
|---|---:|
| Saldo inicial | 100 |
| Saldo final | 150 |
| Suma de beneficios y pérdidas | 50 |
| Saldo final menos saldo inicial | 50 |

La comparación devuelve `True`: ambos cálculos coinciden.

No sumamos `balance_after`, porque cada saldo ya incluye el dinero inicial
y los cambios acumulados. Sumar los saldos contaría repetidamente
el mismo dinero.

Esta sesión terminó con un beneficio neto de 50. Este resultado aislado
no demuestra que la estrategia sea rentable en general.

In [48]:
initial_balance = results["balance_before"].iloc[0]
final_balance = results["balance_after"].iloc[-1]
total_profit = results["profit"].sum()
balance_difference = final_balance - initial_balance

print("Initial balance:", initial_balance)
print("Final balance:", final_balance)
print("Total profit:", total_profit)
print("Balance difference:", balance_difference)
print("Profits match balance change:", total_profit == balance_difference)

Initial balance: 100
Final balance: 150
Total profit: 50
Balance difference: 50
Profits match balance change: True
